# 07 · Recuperación por metadatos y palabras clave (sin OCR)

**Objetivo:** construir un primer buscador documental sobre el catálogo RDF de Memoria Chilena. Usa título, autor, notas, materias, unidades temáticas y conexiones de catalogación. No necesita los PDF ni procesa OCR.

- Busca **las diez fichas principales**. Los cientos de registros RDF relacionados no se confunden con documentos del corpus.
- Combina TF-IDF de palabras y de grupos de caracteres, para tolerar algunas variantes ortográficas.
- Devuelve vínculos a las fichas de origen y relaciones de *catalogación* (compartir una unidad temática no demuestra una relación histórica).
- Como opción, ofrece **pistas de páginas** usando únicamente texto nativo previamente extraído. Esas pistas NO son citas verificadas ni evidencias históricas aprobadas.
- Genera automáticamente `data/output/etapa7.zip`. No descarga archivos ni requiere ejecutar las etapas 1–6 otra vez.

**Ejecución:** abre este notebook en `notebooks/` y elige *Run → Run All Cells*. El proyecto incluye el `etapa4.zip` que necesita.

In [1]:
# Instala solamente los paquetes que falten en este kernel de Jupyter.
import importlib.util, subprocess, sys
requeridos = {"numpy": "numpy", "pandas": "pandas", "rdflib": "rdflib", "sklearn": "scikit-learn"}
faltan = [paquete for modulo, paquete in requeridos.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando dependencias faltantes:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *faltan])
else:
    print("Dependencias listas: no se necesita instalar nada.")

Dependencias listas: no se necesita instalar nada.


In [2]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import io, json, re, unicodedata, itertools
from datetime import datetime, timezone
from urllib.parse import unquote

import numpy as np
import pandas as pd
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import DC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

WORK = Path.cwd().resolve()
ROOT = WORK.parent if WORK.name.lower() == "notebooks" else WORK
ARCHIVO = ROOT / "data" / "output" / "etapa4.zip"
if not ARCHIVO.is_file():
    ARCHIVO = ROOT / "data" / "input" / "etapa4.zip"
assert ARCHIVO.is_file(), f"No encuentro etapa4.zip en el proyecto: {ROOT}"
SALIDA = ROOT / "data" / "output" / "etapa7"
SALIDA.mkdir(parents=True, exist_ok=True)

with ZipFile(ARCHIVO) as zip4:
    inv = pd.read_csv(io.BytesIO(zip4.read("inventario_derechos.csv"))).fillna("")
    ttl_bytes = zip4.read("grafo_corpus_principal.ttl")
    informe4 = json.loads(zip4.read("informe_etapa4.json").decode("utf-8"))

G = Graph()
G.parse(data=ttl_bytes.decode("utf-8"), format="turtle")
assert len(inv) == 10 and len(G) == 831, "El inventario RDF no coincide con la auditoría conocida"
print("Inventario:",len(inv),"documentos | Grafo principal:",len(G),"triples")
print("Sin OCR, sin conexión de red y sin descargar PDFs")

Inventario: 10 documentos | Grafo principal: 831 triples
Sin OCR, sin conexión de red y sin descargar PDFs


## Metadatos indexables y procedencia

Las palabras clave provienen **de campos documentados**: títulos, autores, notas, materias y unidades RDF. No se infieren personas ni acontecimientos del texto histórico.

In [3]:
P_MC = "http://www.memoriachilena.cl/schemas/"

def literal(s, prop):
    return " | ".join(sorted({str(o) for o in G.objects(s, prop) if isinstance(o, Literal) and str(o).strip()}))

def humanizar_unidad(uri):
    return unquote(str(uri).rsplit(":",1)[-1]).replace("_", " ")

filas=[]
for _,r in inv.iterrows():
    ficha=str(r["url"])
    s=URIRef(ficha)
    assert (s,DC.title,None) in G, f"No encuentro ficha en grafo principal: {ficha}"
    unidades=sorted({humanizar_unidad(o) for o in G.objects(s, URIRef("urn:mcgraphrag:inCatalogUnit"))})
    materias=sorted({str(o).rsplit(":",1)[-1] for o in G.objects(s,URIRef("urn:mcgraphrag:hasCatalogSubject"))})
    notas=literal(s,URIRef(P_MC+"notas"))
    datos=literal(s,URIRef(P_MC+"datos"))
    filas.append({
        "mcid":str(r["mcid_rdf"]),"ficha":ficha,
        "titulo":literal(s,DC.title),"autor":literal(s,DC.creator),
        "fecha":str(r["fecha_rdf"]),
        "materias":" | ".join(materias),"unidades":" | ".join(unidades),
        "notas":notas,"datos":datos,
        "derechos":str(r["derechos_declarados"]),
        "rdf_fuente":str(r["rdf_url"]),
    })
catalogo=pd.DataFrame(filas).sort_values("mcid").reset_index(drop=True)
assert catalogo.mcid.nunique()==10
# Solo facetas catalográficas explícitas: útil para filtros y para explicar por qué se recuperó un documento.
display(catalogo[["mcid","titulo","autor","materias","ficha"]])

,mcid,titulo,autor,materias,ficha
0,MC0014753,Doctrina christiana y catecismo para la instru...,,Cultura | Etnias | Historia,https://www.memoriachilena.gob.cl/602/w3-artic...
1,MC0014754,Parte primera de la Chrónica del Perú : que tr...,"Cieza de León, Pedro de, ca. 1520-1554",Cultura | Historia,https://www.memoriachilena.gob.cl/602/w3-artic...
2,MC0014755,"Primera parte de los commentarios reales, que ...","Vega, Garcilaso de la",Cultura | Historia,https://www.memoriachilena.gob.cl/602/w3-artic...
3,MC0014757,Historia natural y moral de las Indias : en qu...,"Acosta, José de",Cultura | Historia,https://www.memoriachilena.gob.cl/602/w3-artic...
4,MC0014760,Extirpación de la idolatria del Piru,"Arriaga, Pablo José de",Cultura | Etnias | Historia,https://www.memoriachilena.gob.cl/602/w3-artic...
5,MC0014761,Vocabulario de la lengua aymara,"Bertonio, Ludovico",Cultura | Etnias | Historia,https://www.memoriachilena.gob.cl/602/w3-artic...
6,MC0023126,La Tirana de Tarapacá,"Uribe Echevarría, Juan, 1908-1988",Cultura%20 | Historia,https://www.memoriachilena.gob.cl/602/w3-artic...
7,MC0023128,El Altiplano de Arica,"Niemeyer Fernández, Hans, 1921-2005",Cultura%20 | Historia,https://www.memoriachilena.gob.cl/602/w3-artic...
8,MC0038283,Con mi humilde devoción : bailes chinos en Chi...,"Mercado, Claudio | Rondón, Víctor",Cultura | Historia | Sociedad,https://www.memoriachilena.gob.cl/602/w3-artic...
9,MC0038290,El término Ylla y su potencial simbólico en el...,"Manríquez Soto, Viviana",Cultura%20 | Historia | Sociedad,https://www.memoriachilena.gob.cl/602/w3-artic...


## Palabras clave y búsqueda flexible

Cada campo se pondera de forma transparente: títulos y notas tienen más peso que las unidades temáticas comunes. Los sinónimos de consulta solo sirven para mejorar la recuperación; **no se convierten en relaciones del grafo**.

In [4]:
STOPWORDS = set("""de del la las el los lo un una unos unas y e o u para por en con sin sobre al a ante bajo desde durante entre hacia hasta que se su sus le les lo como donde que cual cuales quien quienes es son fue fueron esta este esto estos estas mas muy ya mas no pero pues dos tres parte primera segundo primero tratado historia natural libro obras las cosas et el i y""".split())

def norm(x):
    x=unicodedata.normalize("NFKD",str(x).lower())
    return "".join(c for c in x if not unicodedata.combining(c))

def piezas(c):
    # Los límites de campos se conservan para facilitar el diagnóstico.
    return " ".join([
      (str(c.titulo)+" ")*4,
      (str(c.autor)+" ")*2,
      (str(c.notas)+" ")*2,
      str(c.datos),
      (str(c.materias)+" ")*2,
      str(c.unidades)
    ])

catalogo["texto_indice"] = catalogo.apply(piezas,axis=1)
vec_pal=TfidfVectorizer(strip_accents="unicode", lowercase=True, ngram_range=(1,2),
                        stop_words=sorted(STOPWORDS), token_pattern=r"(?u)\b\w\w+\b",sublinear_tf=True)
vec_car=TfidfVectorizer(strip_accents="unicode",lowercase=True,analyzer="char_wb",ngram_range=(3,5),sublinear_tf=True)
Xpal=vec_pal.fit_transform(catalogo.texto_indice)
Xcar=vec_car.fit_transform(catalogo.texto_indice)

# Equivalencias para búsqueda: son ayudas lexicográficas, no afirmaciones históricas.
SINONIMOS={
 "quechua":"quichua", "quichua":"quechua",
 "aymara":"aimara", "aimara":"aymara",
 "inca":"ynca incas yncas", "incas":"yncas incas",
 "peru":"piru", "piru":"peru",
 "evangelizacion":"catecismo doctrina misiones", "catecismo":"doctrina",
 "idolatria":"idolos extirpacion", "rituales":"ritos ceremonias religiosidad",
}

def ampliar(q):
    tokens=re.findall(r"[a-z0-9]+",norm(q))
    nuevos=[SINONIMOS[t] for t in tokens if t in SINONIMOS]
    return q+" "+" ".join(nuevos)

def coincidencias(fila,q):
    términos=[w for w in re.findall(r"[a-z0-9]+",norm(q)) if w not in STOPWORDS and len(w)>2]
    campos={"título":str(fila.titulo),"autor":str(fila.autor),"notas":str(fila.notas),
            "materias":str(fila.materias),"unidades":str(fila.unidades),"datos":str(fila.datos)}
    return ", ".join(k for k,v in campos.items() if any(t in norm(v) for t in términos))

def buscar(query,top_k=6):
    q=ampliar(query)
    sim_pal=cosine_similarity(vec_pal.transform([q]),Xpal).ravel()
    sim_car=cosine_similarity(vec_car.transform([q]),Xcar).ravel()
    score=0.7*sim_pal+0.3*sim_car
    orden=np.argsort(-score)[:top_k]
    resultado=catalogo.iloc[orden][["mcid","titulo","autor","fecha","materias","unidades","ficha"]].copy()
    resultado.insert(1,"score",np.round(score[orden],3))
    resultado["coincide_en"]=[coincidencias(catalogo.iloc[i],query) for i in orden]
    resultado["evidencia"]="solo metadatos catalográficos"
    return resultado.reset_index(drop=True)

# Extrae palabras descriptivas como apoyo a la navegación, con su origen editorial intacto.
from sklearn.feature_extraction.text import TfidfVectorizer as TF
v_claves=TF(strip_accents="unicode",lowercase=True,ngram_range=(1,2),
            stop_words=sorted(STOPWORDS),token_pattern=r"(?u)\b\w\w\w+\b",max_features=5000)
metadata_sin_repeticion=(catalogo.titulo+" "+catalogo.notas+" "+catalogo.materias+" "+catalogo.unidades)
K=v_claves.fit_transform(metadata_sin_repeticion)
ter=v_claves.get_feature_names_out()
keywords=[]
for i,doc in catalogo.iterrows():
    row=K.getrow(i)
    ix=np.argsort(-row.data)[:12]
    for pos in ix:
        keywords.append({"mcid":doc.mcid,"palabra_clave":ter[row.indices[pos]],
                         "peso_tfidf":round(float(row.data[pos]),4),
                         "procedencia":"campos catalográficos; extracción automática"})
claves=pd.DataFrame(keywords)
display(claves.groupby("mcid").head(5).reset_index(drop=True))

,mcid,palabra_clave,peso_tfidf,procedencia
0,MC0014753,quichua,0.2501,campos catalográficos; extracción automática
1,MC0014753,quichua aymara,0.2501,campos catalográficos; extracción automática
2,MC0014753,aymara,0.2126,campos catalográficos; extracción automática
3,MC0014753,doctrina,0.1250,campos catalográficos; extracción automática
4,MC0014753,compuesto,0.1250,campos catalográficos; extracción automática
5,MC0014754,chronica,0.1644,campos catalográficos; extracción automática
6,MC0014754,provincias,0.1644,campos catalográficos; extracción automática
7,MC0014754,demarcacion,0.1644,campos catalográficos; extracción automática
8,MC0014754,tracta,0.1644,campos catalográficos; extracción automática
9,MC0014754,nuevas,0.1644,campos catalográficos; extracción automática


## Consultas de ejemplo

Puedes reemplazar el texto por cualquier tema. La primera tabla explica por qué aparecen los documentos, sin atribuirles afirmaciones históricas no verificadas.

In [5]:
CONSULTA = "evangelización y lenguas indígenas en los Andes"
res=buscar(CONSULTA,top_k=6)
print("Consulta:",CONSULTA)
display(res[["mcid","score","titulo","coincide_en","ficha"]])

for ejemplo in ["vocabulario lengua aymara", "extirpación de idolatría", "cronistas del Perú", "La Tirana Tarapacá"]:
    r=buscar(ejemplo,top_k=3)
    print("\n",ejemplo,"→", " | ".join(r.mcid.tolist()))

Consulta: evangelización y lenguas indígenas en los Andes


,mcid,score,titulo,coincide_en,ficha
0,MC0014753,0.303,Doctrina christiana y catecismo para la instru...,notas,https://www.memoriachilena.gob.cl/602/w3-artic...
1,MC0014761,0.041,Vocabulario de la lengua aymara,,https://www.memoriachilena.gob.cl/602/w3-artic...
2,MC0014754,0.032,Parte primera de la Chrónica del Perú : que tr...,,https://www.memoriachilena.gob.cl/602/w3-artic...
3,MC0014760,0.024,Extirpación de la idolatria del Piru,,https://www.memoriachilena.gob.cl/602/w3-artic...
4,MC0014757,0.022,Historia natural y moral de las Indias : en qu...,,https://www.memoriachilena.gob.cl/602/w3-artic...
5,MC0014755,0.018,"Primera parte de los commentarios reales, que ...",,https://www.memoriachilena.gob.cl/602/w3-artic...



 vocabulario lengua aymara → MC0014761 | MC0014753 | MC0014760

 extirpación de idolatría → MC0014760 | MC0014755 | MC0014754

 cronistas del Perú → MC0014760 | MC0014755 | MC0014754

 La Tirana Tarapacá → MC0023126 | MC0038290 | MC0014754


## Exploración de relaciones catalográficas

Una arista de este grafo significa únicamente que dos fichas comparten una **unidad temática del catálogo**. No es evidencia de influencia, contacto o causalidad entre sus autores.

In [6]:
from collections import Counter
unidades={r.mcid:set(str(r.unidades).split(" | "))-{''} for r in catalogo.itertuples()}
frecuencia=Counter(u for conjunto in unidades.values() for u in conjunto)
relaciones=[]
for a,b in itertools.combinations(catalogo.mcid.tolist(),2):
    comunes=sorted(unidades[a] & unidades[b])
    if comunes:
        # Reduce el peso de las unidades muy frecuentes, pero no altera el hecho catalográfico.
        score=sum(1/frecuencia[u] for u in comunes)
        relaciones.append({"mcid_origen":a,"mcid_destino":b,"unidades_comunes":" | ".join(comunes),
                          "n_unidades_comunes":len(comunes),"peso_por_especificidad":round(score,4),
                          "tipo":"comparten unidad temática catalográfica"})
red=pd.DataFrame(relaciones).sort_values("peso_por_especificidad",ascending=False)
print("Enlaces entre pares de fichas según unidades temáticas compartidas:",len(red))
display(red.head(10))

Enlaces entre pares de fichas según unidades temáticas compartidas: 45


,mcid_origen,mcid_destino,unidades_comunes,n_unidades_comunes,peso_por_especificidad,tipo
4,MC0014753,MC0014761,elpuebloaymara | lasmisionesjesuitasdeparaguay...,3,0.80,comparten unidad temática catalográfica
39,MC0023126,MC0023128,geografiaculturaldechile | religiosidadandina,2,0.60,comparten unidad temática catalográfica
17,MC0014755,MC0014757,cronistasperuanos | elpuebloaymara | religiosi...,3,0.55,comparten unidad temática catalográfica
18,MC0014755,MC0014760,cronistasperuanos | elpuebloaymara | religiosi...,3,0.55,comparten unidad temática catalográfica
24,MC0014757,MC0014760,cronistasperuanos | elpuebloaymara | religiosi...,3,0.55,comparten unidad temática catalográfica
11,MC0014754,MC0014760,cronistasperuanos | religiosidadandina,2,0.35,comparten unidad temática catalográfica
9,MC0014754,MC0014755,cronistasperuanos | religiosidadandina,2,0.35,comparten unidad temática catalográfica
10,MC0014754,MC0014757,cronistasperuanos | religiosidadandina,2,0.35,comparten unidad temática catalográfica
25,MC0014757,MC0014761,elpuebloaymara | religiosidadandina,2,0.30,comparten unidad temática catalográfica
1,MC0014753,MC0014755,elpuebloaymara | religiosidadandina,2,0.30,comparten unidad temática catalográfica


## Opcional: pistas de páginas con texto ya extraído

Esta exploración NO hace OCR. Puede localizar páginas candidatas en los seis documentos para los que ya existe extracción de texto nativo. **Como su texto presenta ruido, úsala solo para elegir páginas que luego se revisarán visualmente**, nunca para producir citas literales automáticamente.

In [7]:
with ZipFile(ARCHIVO) as z:
    fragmentos = pd.read_csv(io.BytesIO(z.read("fragmentos.csv")),usecols=["mcid","pagina_pdf","texto","ficha"]).fillna("")

def paginas_posibles(query, top_docs=4, top_paginas=8):
    prim=buscar(query,top_k=top_docs)
    ids=prim.mcid.tolist()
    vocab=[t for t in re.findall(r"[a-z0-9]+",norm(ampliar(query))) if len(t)>=4 and t not in STOPWORDS]
    resultados=[]
    frag=fragmentos[fragmentos.mcid.isin(ids)]
    for (mcid,pagina),group in frag.groupby(["mcid","pagina_pdf"]):
        txt=norm(" ".join(group.texto.astype(str)))
        hallados=sorted({t for t in vocab if t in txt})
        if hallados:
            resultados.append({"mcid":mcid,"pagina_pdf":int(pagina),"terminos_encontrados":", ".join(hallados),
                               "n_terminos":len(hallados),"ficha":group.ficha.iloc[0],
                               "advertencia":"pista, texto nativo sin validar"})
    return pd.DataFrame(resultados).sort_values(["n_terminos","mcid"],ascending=[False,True]).head(top_paginas) if resultados else pd.DataFrame(columns=["mcid","pagina_pdf","terminos_encontrados","ficha"])

print("Búsqueda de páginas candidatas (sin OCR, sin citas automáticas):")
display(paginas_posibles("catecismo aymara"))

Búsqueda de páginas candidatas (sin OCR, sin citas automáticas):


,mcid,pagina_pdf,terminos_encontrados,n_terminos,ficha,advertencia
0,MC0014753,128,catecismo,1,https://www.memoriachilena.gob.cl/602/w3-artic...,"pista, texto nativo sin validar"
1,MC0014753,140,catecismo,1,https://www.memoriachilena.gob.cl/602/w3-artic...,"pista, texto nativo sin validar"
2,MC0014755,146,aymara,1,https://www.memoriachilena.gob.cl/602/w3-artic...,"pista, texto nativo sin validar"
3,MC0014755,354,doctrina,1,https://www.memoriachilena.gob.cl/602/w3-artic...,"pista, texto nativo sin validar"
4,MC0014761,15,aymara,1,https://www.memoriachilena.gob.cl/602/w3-artic...,"pista, texto nativo sin validar"
5,MC0014761,16,aymara,1,https://www.memoriachilena.gob.cl/602/w3-artic...,"pista, texto nativo sin validar"
6,MC0014761,316,aymara,1,https://www.memoriachilena.gob.cl/602/w3-artic...,"pista, texto nativo sin validar"
7,MC0014761,372,aymara,1,https://www.memoriachilena.gob.cl/602/w3-artic...,"pista, texto nativo sin validar"


## Exportación automática

Se guardan únicamente metadatos, palabras clave, relaciones de catálogo y resultados de ejemplo. No se adjuntan PDFs ni transcripciones de páginas. Esto también facilita subir el código a GitHub sin redistribuir los libros.

In [8]:
catalogo.drop(columns=["texto_indice"]).to_csv(SALIDA/"catalogo_para_busqueda.csv",index=False,encoding="utf-8-sig")
claves.to_csv(SALIDA/"palabras_clave.csv",index=False,encoding="utf-8-sig")
red.to_csv(SALIDA/"relaciones_catalograficas.csv",index=False,encoding="utf-8-sig")
examples=[]
for consulta in ["catecismo y quechua", "vocabulario aymara", "extirpación de idolatría", "historia de los incas", "La Tirana Tarapacá"]:
    x=buscar(consulta,top_k=5)
    x.insert(0,"consulta",consulta)
    examples.append(x)
pd.concat(examples,ignore_index=True).to_csv(SALIDA/"consultas_demostracion.csv",index=False,encoding="utf-8-sig")
informe={
 "fecha_utc":datetime.now(timezone.utc).isoformat(),
 "proyecto":"Memoria Chilena GraphRAG", "etapa":7,
 "metodo":"recuperacion por metadatos RDF, TF-IDF palabra+caracteres, palabras clave y co-membresia tematica",
 "requiere_ocr":False,"requiere_pdf":False,"requiere_internet":False,
 "fichas_indexadas":len(catalogo),"triples_corpus_principal":len(G),
 "palabras_clave_extraidas":len(claves),"pares_de_documentos_con_unidad_comun":len(red),
 "advertencias":["solo recuperación catalográfica; no lectura de contenido histórico", "las relaciones son curatoriales, no históricas", "las pistas de páginas proceden de texto sin validación y NO son citas", "las materias y palabras clave provienen del catálogo, no de un OCR nuevo"]}
(SALIDA/"informe_etapa7.json").write_text(json.dumps(informe,ensure_ascii=False,indent=2),encoding="utf-8")
ZIP_FINAL=ROOT/"data"/"output"/"etapa7.zip"
with ZipFile(ZIP_FINAL,"w",ZIP_DEFLATED) as z:
    for f in sorted(SALIDA.iterdir()):
        if f.is_file(): z.write(f,arcname=f.name)
assert ZIP_FINAL.is_file()
with ZipFile(ZIP_FINAL) as z:
    assert {"catalogo_para_busqueda.csv","palabras_clave.csv","relaciones_catalograficas.csv","consultas_demostracion.csv","informe_etapa7.json"}.issubset(set(z.namelist()))
assert buscar("vocabulario lengua aymara",1).iloc[0].mcid=="MC0014761"
assert buscar("extirpación de la idolatría",1).iloc[0].mcid=="MC0014760"
print("OK ·",len(catalogo),"fichas indexadas y",len(claves),"palabras clave ponderadas")
print("ZIP creado:",ZIP_FINAL)

OK · 10 fichas indexadas y 114 palabras clave ponderadas
ZIP creado: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa7.zip
